# Persian AI-Text Detection — Unified (Colab) Retraining Pipeline

This single notebook supersedes the former `colab_retrain.ipynb` and
`colab_jev_parsbert.ipynb` (and the legacy root `main.ipynb`, whose
dataset-building step now lives in `training/build_dataset.py`). It trains
everything the project deploys, in order:

1. classic ML (SVM, Logistic Regression, Naive Bayes, Random Forest, Gradient
   Boosting) on a capped unigram-to-trigram TF-IDF space with real
   hyperparameter search,
2. BiLSTM (from scratch, GPU),
3. GPT-2 (full fine-tune of all 12 blocks, mixed precision),
4. ParsBERT (bidirectional encoder fine-tune with `[CLS]` pooling),
5. the System One decision model (typed decision heads distilled from the deployed
   TF-IDF + Logistic Regression teacher) — **legacy GPT-2-backed recipe**;
   the deployed ParsBERT-backed System One v2 is trained locally with
   `python training/train_system_one.py` instead.

Everything shared with the serving stack — the shekar preprocessing
pipelines, the stylometric feature formulas, the model architectures, and the
System One typed-question schema — is **imported from the `aitextd` package**
instead of redefined here, so train/serve drift is impossible by
construction. The train/val/test split matches `training/split.py` (seed 42,
80/20 stratified test), so metrics stay comparable to `models.json`.

**Runtime:** `Runtime > Change runtime type > GPU` before running — the
BiLSTM/GPT-2/ParsBERT/System One sections use it; everything else is CPU-bound.

## What to put in Google Drive first

Create `MyDrive/AITextDetection/` with:

| path | why |
| --- | --- |
| `processedDataset.csv` | the balanced 21,092-row dataset all models train on |
| `tfidf_vectorizer.joblib` | the **deployed** vectorizer (System One teacher features must match it) |
| `logregModel.joblib` | the **deployed** logistic regression System One distils soft targets from |
| `models.json` | read only to carry current `ensembleWeight` values forward |
| `repo/aitextd/` + `repo/training/` | the project source folders this notebook imports |

Output is written to `MyDrive/AITextDetection/trained/` as each model
finishes (a runtime disconnect can't lose finished work), and the last cell
zips the whole set for download. To deploy: drop the artifacts into the
project's `models/` directory and merge the `models.json` fragment.


## 1. Setup

In [ ]:
!pip install -q shekar "transformers>=4.40" scikit-learn joblib

import torch

print("torch:", torch.__version__, "| transformers:", __import__("transformers").__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU — go to Runtime > Change runtime type > GPU before continuing.")

## 2. Mount Google Drive and locate the inputs

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/AITextDetection"
OUT_DIR = os.path.join(DRIVE_DIR, "trained")
os.makedirs(OUT_DIR, exist_ok=True)

REQUIRED = {
    "processedDataset.csv": "the dataset every model trains on",
    "tfidf_vectorizer.joblib": "the deployed vectorizer System One's teacher features must match",
    "logregModel.joblib": "the deployed logistic regression System One distils soft targets from",
    "repo/aitextd": "the project package this notebook imports",
    "repo/training": "project training helpers this notebook imports",
}
missing = [
    f"{name}  ({why})" for name, why in REQUIRED.items()
    if not os.path.exists(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        "Copy these into " + DRIVE_DIR + ":\n  - " + "\n  - ".join(missing)
    )
print("Drive ready. Output ->", OUT_DIR)

In [ ]:
import json
import math
import multiprocessing
import os
import re
import shutil
import sys
import time
from collections import Counter

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from joblib import Parallel, delayed
from scipy.sparse import csr_matrix, hstack
from scipy.stats import loguniform, randint
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Shared train/serve logic comes from the project package -- NOT redefined here.
sys.path.insert(0, "/content/drive/MyDrive/AITextDetection/repo")
sys.path.insert(0, "/content/drive/MyDrive/AITextDetection/repo/training")

from aitextd.architectures import (
    BiLSTMClassifier,
    GPT2Classifier,
    SystemOneDecisionModel,
    ParsBERTClassifier,
)
from aitextd.constants import SYSTEM_ONE_CHOICE_CANDIDATES, SYSTEM_ONE_SCORE_LEVELS
from aitextd.features import STYLE_FEATURES_VERSION, STYLE_FEATURE_NAMES, compute_style_features
from aitextd.text import normalize_for_pretrained, tokenize_persian_text
from prepare_system_one_data import (
    CHOICE_CLIP,
    LABEL_WEIGHT,
    NOUL_BAND,
    SCORE_SIGMA,
    TEACHER_WEIGHT,
    choice_target,
    noul_target,
    score_target,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("Using device:", DEVICE)

## 3. Load the dataset

`processedDataset.csv` is the already balanced and shuffled dataset the
deployed models were trained on, so the numbers below are directly comparable
to the ones in `models.json`. If you'd rather rebuild it from the raw source
files, run `python training/build_dataset.py` locally (the logic that used to
live in the legacy `main.ipynb`).

In [ ]:
df = pd.read_csv(os.path.join(DRIVE_DIR, "processedDataset.csv"))
print(df.shape)
print(df["aiGenerated"].value_counts())
df.head(3)

## 4. Train / validation / test split

Same seed and 80/20 stratified test split as `training/split.py`, so
`test_idx` lines up with the held-out set the currently-deployed models were
scored on. The 20% test slice is touched **exactly once**, at the end, per
model; the 10% validation slice carved out of the training portion is what
early stopping uses.

In [ ]:
TEST_SIZE = 0.2
VAL_SIZE = 0.1  # fraction of the train+val portion

idx = df.index.to_numpy()
y_all = df["aiGenerated"].to_numpy()

train_val_idx, test_idx = train_test_split(
    idx, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_all[train_val_idx],
)

print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")
y_train, y_val, y_test = y_all[train_idx], y_all[val_idx], y_all[test_idx]

## 5. Preprocessing (lemmatize + tokenize, for the TF-IDF / BiLSTM models)

Same shekar pipeline the production API uses — imported from `aitextd.text`
(`tokenize_persian_text`) instead of redefined. This is the slow,
CPU-bound step; the result is cached to Drive so you only pay it once.

(The pretrained models use a *different*, lighter preprocessing — section 9.)

In [ ]:
TOKEN_CACHE_PATH = os.path.join(DRIVE_DIR, "tokenized_texts_colab.joblib")

try:
    tokenized = joblib.load(TOKEN_CACHE_PATH)
    assert len(tokenized) == len(df) and isinstance(tokenized[0], list)
    print(f"Loaded cached tokenization for {len(tokenized)} documents.")
except (FileNotFoundError, AssertionError):
    print(f"Tokenizing {len(df)} documents on {multiprocessing.cpu_count()} cores (the slow part)...")
    tokenized = Parallel(n_jobs=multiprocessing.cpu_count())(
        delayed(tokenize_persian_text)(text) for text in tqdm(df["text"])
    )
    joblib.dump(tokenized, TOKEN_CACHE_PATH)
    print("Cached to Drive for next time.")

cleaned_texts = [" ".join(tokens) for tokens in tokenized]

## 6. TF-IDF features

Unigrams through trigrams with a capped vocabulary — far smaller and more
robust than the deployed 9.3M-dimension trigram-only space.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=150_000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
    strip_accents="unicode",
)

train_texts = [cleaned_texts[i] for i in train_idx]
tfidf_vectorizer.fit(train_texts)

X_all = tfidf_vectorizer.transform(cleaned_texts)
X_train, X_val, X_test = X_all[train_idx], X_all[val_idx], X_all[test_idx]
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))
print(X_train.shape, X_val.shape, X_test.shape)

# Also build a train+val combined matrix for the final classic-model refits
# (once hyperparameters are chosen via CV on train only, refit on train+val
# before the single held-out test evaluation).
trainval_idx = np.concatenate([train_idx, val_idx])
y_trainval = y_all[trainval_idx]
X_trainval = X_all[trainval_idx]

## 6b. Stylometric text-style features

Imported byte-for-byte from `aitextd.features` (the same functions the server
calls), so served predictions can never drift from what these models were
trained on. Hstacked onto the TF-IDF matrix for every classic ML model below.

In [ ]:
STYLE_CACHE_PATH = os.path.join(DRIVE_DIR, "style_features_colab.joblib")

try:
    payload = joblib.load(STYLE_CACHE_PATH)
    assert payload["version"] == STYLE_FEATURES_VERSION
    style_matrix = payload["matrix"]
    assert style_matrix.shape == (len(df), len(STYLE_FEATURE_NAMES))
    print(f"Loaded cached style features for {style_matrix.shape[0]} documents.")
except (FileNotFoundError, KeyError, TypeError, IndexError, AssertionError):
    print(f"Computing style features for {len(df)} documents (single-process, shared vectorizer)...")
    style_matrix = np.array([
        [compute_style_features(str(text), cleaned, tfidf_vectorizer)[name] for name in STYLE_FEATURE_NAMES]
        for text, cleaned in tqdm(zip(df["text"], cleaned_texts), total=len(df))
    ])
    joblib.dump({"version": STYLE_FEATURES_VERSION, "matrix": style_matrix}, STYLE_CACHE_PATH)
    print("Cached to Drive for next time.")

# Style columns always come after the TF-IDF columns -- TFIDF_DIM marks
# where they start, used below to keep them out of the tree models' chi2
# selection (they should always be kept, not compete for a chi2 slot).
TFIDF_DIM = X_all.shape[1]
X_all = hstack([X_all, csr_matrix(style_matrix)]).tocsr()
X_train, X_val, X_test = X_all[train_idx], X_all[val_idx], X_all[test_idx]
X_trainval = X_all[trainval_idx]
print("TF-IDF dim:", TFIDF_DIM, "| + style features ->", X_all.shape[1])

## 7. Evaluation helper

In [ ]:
results = {}  # model_id -> {"model": ..., "metrics": ..., "confusion": ..., "extra": {...}}

def evaluate(model, X, y):
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    metrics = {
        "accuracy": round(float(accuracy_score(y, y_pred)), 4),
        "precision": round(float(precision_score(y, y_pred)), 4),
        "recall": round(float(recall_score(y, y_pred)), 4),
        "f1Score": round(float(f1_score(y, y_pred)), 4),
    }
    confusion = {
        "trueNegative": int(tn),
        "falsePositive": int(fp),
        "falseNegative": int(fn),
        "truePositive": int(tp),
    }
    return metrics, confusion

## 8. SVM (LinearSVC)

In [ ]:
svm_search = RandomizedSearchCV(
    LinearSVC(random_state=RANDOM_SEED, max_iter=10_000, dual=True),
    param_distributions={
        "C": loguniform(1e-3, 1e2),
        "loss": ["hinge", "squared_hinge"],
        "class_weight": [None, "balanced"],
    },
    n_iter=15,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
)
t0 = time.time()
svm_search.fit(X_train, y_train)
print(f"Best params: {svm_search.best_params_}  (searched in {time.time()-t0:.0f}s)")

svm_model = LinearSVC(random_state=RANDOM_SEED, max_iter=10_000, **svm_search.best_params_)
svm_model.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(svm_model, X_test, y_test)
print(metrics)
results["svm"] = {"model": svm_model, "metrics": metrics, "confusion": confusion,
                   "extra": {"algorithm": "LinearSVC", "best_params": svm_search.best_params_}}

## 9. Logistic Regression (the System One teacher)

In [ ]:
logreg_search = RandomizedSearchCV(
    LogisticRegression(random_state=RANDOM_SEED, max_iter=3000),
    param_distributions={
        "C": loguniform(1e-3, 1e2),
        "solver": ["liblinear", "saga"],
        "class_weight": [None, "balanced"],
    },
    n_iter=15,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
)
t0 = time.time()
logreg_search.fit(X_train, y_train)
print(f"Best params: {logreg_search.best_params_}  (searched in {time.time()-t0:.0f}s)")

logreg_model = LogisticRegression(random_state=RANDOM_SEED, max_iter=3000, **logreg_search.best_params_)
logreg_model.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(logreg_model, X_test, y_test)
print(metrics)
results["logreg"] = {"model": logreg_model, "metrics": metrics, "confusion": confusion,
                      "extra": {"algorithm": "LogisticRegression", "best_params": logreg_search.best_params_}}

## 10. Naive Bayes

In [ ]:
nb_search = GridSearchCV(
    MultinomialNB(),
    param_grid={"alpha": [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0]},
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
)
nb_search.fit(X_train, y_train)
print("Best params:", nb_search.best_params_)

nb_model = MultinomialNB(**nb_search.best_params_)
nb_model.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(nb_model, X_test, y_test)
print(metrics)
results["naive_bayes"] = {"model": nb_model, "metrics": metrics, "confusion": confusion,
                           "extra": {"algorithm": "MultinomialNB", "best_params": nb_search.best_params_}}

## 11. Random Forest

Trees don't scale to 150k dimensions, so chi2 selects the top TF-IDF
features (style columns always pass through untouched), and tree depth /
leaf size are tuned — unlimited-depth trees overfit noisy TF-IDF features.

In [ ]:
TREE_FEATURE_COUNT = 30_000

tree_selector = ColumnTransformer(
    [
        ("tfidf_select", SelectKBest(chi2, k=TREE_FEATURE_COUNT), slice(0, TFIDF_DIM)),
        ("style", "passthrough", slice(TFIDF_DIM, None)),
    ],
    sparse_threshold=1.0,
)
tree_selector.fit(X_train, y_train)

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1),
    param_distributions={
        "n_estimators": randint(200, 500),
        "max_depth": [15, 25, 40, None],
        "min_samples_leaf": randint(1, 8),
        "max_features": ["sqrt", "log2"],
        "class_weight": [None, "balanced"],
    },
    n_iter=12,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=2,
)
t0 = time.time()
rf_search.fit(tree_selector.transform(X_train), y_train)
print(f"Best params: {rf_search.best_params_}  (searched in {time.time()-t0:.0f}s)")

rf_pipeline = Pipeline([
    ("select", tree_selector),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1, **rf_search.best_params_)),
])
rf_pipeline.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(rf_pipeline, X_test, y_test)
print(metrics)
results["random_forest"] = {"model": rf_pipeline, "metrics": metrics, "confusion": confusion,
                             "extra": {"algorithm": "RandomForestClassifier", "best_params": rf_search.best_params_,
                                       "featureCount": TREE_FEATURE_COUNT + len(STYLE_FEATURE_NAMES)}}

## 12. Gradient Boosting

In [ ]:
gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_SEED),
    param_distributions={
        "n_estimators": randint(100, 350),
        "max_depth": [2, 3, 4],
        "learning_rate": loguniform(0.02, 0.25),
        "subsample": [0.7, 0.85, 1.0],
    },
    n_iter=10,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=2,
)
t0 = time.time()
gb_search.fit(tree_selector.transform(X_train), y_train)
print(f"Best params: {gb_search.best_params_}  (searched in {time.time()-t0:.0f}s)")

gb_pipeline = Pipeline([
    ("select", tree_selector),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_SEED, **gb_search.best_params_)),
])
gb_pipeline.fit(X_trainval, y_trainval)
metrics, confusion = evaluate(gb_pipeline, X_test, y_test)
print(metrics)
results["gradient_boosting"] = {"model": gb_pipeline, "metrics": metrics, "confusion": confusion,
                                 "extra": {"algorithm": "GradientBoostingClassifier", "best_params": gb_search.best_params_,
                                           "featureCount": TREE_FEATURE_COUNT + len(STYLE_FEATURE_NAMES)}}

## 13. BiLSTM (GPU, from scratch)

Same architecture as the deployed model — imported from
`aitextd.architectures` — trained properly this time: bigger capacity, weight
decay, gradient clipping, and early stopping against the **validation** set
(not the test set).

In [ ]:
MAX_VOCAB_SIZE = 40_000
MIN_FREQ = 2
MAX_LEN = 256
EMBED_DIM = 200
HIDDEN_DIM = 192
BATCH_SIZE = 64
MAX_EPOCHS = 25
PATIENCE = 4
LR = 1e-3
WEIGHT_DECAY = 1e-5
PAD_IDX = 0
UNK_IDX = 1

train_tokens = [tokenized[i] for i in train_idx]
val_tokens = [tokenized[i] for i in val_idx]
test_tokens = [tokenized[i] for i in test_idx]

def build_vocab(token_lists):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
    for word, freq in counter.most_common(MAX_VOCAB_SIZE):
        if freq < MIN_FREQ:
            break
        vocab.setdefault(word, len(vocab))
    return vocab

vocab = build_vocab(train_tokens)
print("Vocab size:", len(vocab))

def encode(tokens, max_len=MAX_LEN):
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    return ids

def to_tensor(token_lists, labels):
    X = torch.tensor([encode(t) for t in token_lists], dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.float32)
    return X, y

X_train_lstm, y_train_lstm = to_tensor(train_tokens, y_train)
X_val_lstm, y_val_lstm = to_tensor(val_tokens, y_val)
X_test_lstm, y_test_lstm = to_tensor(test_tokens, y_test)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


train_loader = DataLoader(TextDataset(X_train_lstm, y_train_lstm), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TextDataset(X_val_lstm, y_val_lstm), batch_size=256, shuffle=False)
test_loader = DataLoader(TextDataset(X_test_lstm, y_test_lstm), batch_size=256, shuffle=False)

lstm_model = BiLSTMClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

def run_eval(loader):
    lstm_model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            probs = torch.sigmoid(lstm_model(xb.to(DEVICE)))
            all_probs.append(probs.cpu().numpy())
            all_labels.append(yb.numpy())
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds = (probs >= 0.5).astype(int)
    return f1_score(labels, preds), accuracy_score(labels, preds)

best_val_f1 = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    lstm_model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = lstm_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    val_f1, val_acc = run_eval(val_loader)
    print(f"epoch {epoch}/{MAX_EPOCHS}  loss={total_loss/len(X_train_lstm):.4f}  "
          f"val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({time.time()-t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in lstm_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs).")
            break

lstm_model.load_state_dict(best_state)

In [ ]:
lstm_model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        probs = torch.sigmoid(lstm_model(xb.to(DEVICE)))
        all_probs.append(probs.cpu().numpy())
        all_labels.append(yb.numpy())
probs = np.concatenate(all_probs)
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, preds)
tn, fp, fn, tp = cm.ravel()
lstm_metrics = {
    "accuracy": round(float(accuracy_score(y_test, preds)), 4),
    "precision": round(float(precision_score(y_test, preds)), 4),
    "recall": round(float(recall_score(y_test, preds)), 4),
    "f1Score": round(float(f1_score(y_test, preds)), 4),
}
lstm_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(lstm_metrics)
results["bilstm"] = {
    "metrics": lstm_metrics,
    "confusion": lstm_confusion,
    "extra": {"algorithm": "BiLSTM", "vocab_size": len(vocab)},
}

## 14. GPT-2 — full fine-tune on normalized (not lemmatized) text

The pretrained models see only shekar's `Normalizer` — imported from
`aitextd.text` as `normalize_for_pretrained`, the exact function the server
calls — because lemmatization strips the inflection a pretrained language
model needs. The classifier architecture is imported from
`aitextd.architectures` as well.

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel

GPT2_MODEL_NAME = "HooshvareLab/gpt2-fa"

gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_tokenizer.pad_token = "<pad>"
print("Using fast tokenizer:", gpt2_tokenizer.is_fast)

gpt2_lm = GPT2LMHeadModel.from_pretrained(GPT2_MODEL_NAME)
gpt2_backbone = gpt2_lm.transformer
gpt2_hidden_size = gpt2_backbone.config.n_embd
gpt2_n_layer = gpt2_backbone.config.n_layer
print(f"n_layer={gpt2_n_layer}  n_embd={gpt2_hidden_size}  vocab_size={gpt2_backbone.config.vocab_size}")

In [ ]:
GPT2_NORM_CACHE_PATH = os.path.join(DRIVE_DIR, "normalized_texts_gpt2.joblib")

try:
    normalized_texts = joblib.load(GPT2_NORM_CACHE_PATH)
    assert len(normalized_texts) == len(df)
    print(f"Loaded cached normalization for {len(normalized_texts)} documents.")
except (FileNotFoundError, AssertionError):
    print("Normalizing text for the pretrained models (character-level only, no lemmatization)...")
    normalized_texts = Parallel(n_jobs=multiprocessing.cpu_count())(
        delayed(normalize_for_pretrained)(t) for t in tqdm(df["text"])
    )
    joblib.dump(normalized_texts, GPT2_NORM_CACHE_PATH)
    print("Cached to Drive for next time.")

In [ ]:
GPT2_SEQ_LEN = 128
GPT2_BATCH_SIZE = 32
GPT2_MAX_EPOCHS = 4
GPT2_PATIENCE = 2
GPT2_LR_BACKBONE = 2e-5
GPT2_LR_HEAD = 1e-3

# All 12 transformer blocks unfrozen; the (larger) word/position embedding
# table stays frozen to limit overfitting on 21k rows.
for p in gpt2_backbone.parameters():
    p.requires_grad = False
for block in gpt2_backbone.h:
    for p in block.parameters():
        p.requires_grad = True
for p in gpt2_backbone.ln_f.parameters():
    p.requires_grad = True

gpt2_model = GPT2Classifier(gpt2_backbone, gpt2_hidden_size).to(DEVICE)
trainable_backbone_params = [p for p in gpt2_backbone.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_backbone_params) + sum(p.numel() for p in gpt2_model.head.parameters())
print(f"Trainable parameters: {n_trainable:,}")

In [ ]:
def gpt2_encode_batch(text_batch):
    enc = gpt2_tokenizer(
        list(text_batch),
        return_tensors="pt",
        truncation=True,
        max_length=GPT2_SEQ_LEN,
        padding="max_length",
    )
    return enc["input_ids"], enc["attention_mask"]


class TextLabelDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


def gpt2_collate(batch):
    texts, labels = zip(*batch)
    input_ids, attention_mask = gpt2_encode_batch(texts)
    return input_ids, attention_mask, torch.tensor(labels, dtype=torch.float32)


gpt2_train_texts = [normalized_texts[i] for i in train_idx]
gpt2_val_texts = [normalized_texts[i] for i in val_idx]
gpt2_test_texts = [normalized_texts[i] for i in test_idx]

gpt2_train_loader = DataLoader(
    TextLabelDataset(gpt2_train_texts, y_train), batch_size=GPT2_BATCH_SIZE,
    shuffle=True, collate_fn=gpt2_collate,
)
gpt2_val_loader = DataLoader(
    TextLabelDataset(gpt2_val_texts, y_val), batch_size=64,
    shuffle=False, collate_fn=gpt2_collate,
)
gpt2_test_loader = DataLoader(
    TextLabelDataset(gpt2_test_texts, y_test), batch_size=64,
    shuffle=False, collate_fn=gpt2_collate,
)

In [ ]:
optimizer = torch.optim.AdamW([
    {"params": trainable_backbone_params, "lr": GPT2_LR_BACKBONE},
    {"params": gpt2_model.head.parameters(), "lr": GPT2_LR_HEAD},
])
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)

def gpt2_run_eval(loader):
    gpt2_model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                probs = torch.sigmoid(gpt2_model(input_ids.to(DEVICE), attention_mask.to(DEVICE)))
            all_probs.append(probs.float().cpu().numpy())
            all_labels.append(labels.numpy())
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds = (probs >= 0.5).astype(int)
    return f1_score(labels, preds), accuracy_score(labels, preds)

best_val_f1 = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, GPT2_MAX_EPOCHS + 1):
    t0 = time.time()
    gpt2_model.train()
    total_loss = 0.0
    for i, (input_ids, attention_mask, labels) in enumerate(gpt2_train_loader, start=1):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            logits = gpt2_model(input_ids, attention_mask)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(gpt2_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
        if i % 50 == 0:
            print(f"  epoch {epoch} batch {i}/{len(gpt2_train_loader)} loss={loss.item():.4f} "
                  f"elapsed={time.time()-t0:.0f}s", flush=True)

    val_f1, val_acc = gpt2_run_eval(gpt2_val_loader)
    print(f"Epoch {epoch}/{GPT2_MAX_EPOCHS}  loss={total_loss/len(gpt2_train_texts):.4f}  "
          f"val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({time.time()-t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in gpt2_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= GPT2_PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {GPT2_PATIENCE} epochs).")
            break

gpt2_model.load_state_dict(best_state)

In [ ]:
gpt2_model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for input_ids, attention_mask, labels in gpt2_test_loader:
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            probs = torch.sigmoid(gpt2_model(input_ids.to(DEVICE), attention_mask.to(DEVICE)))
        all_probs.append(probs.float().cpu().numpy())
        all_labels.append(labels.numpy())
probs = np.concatenate(all_probs)
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, preds)
tn, fp, fn, tp = cm.ravel()
gpt2_metrics = {
    "accuracy": round(float(accuracy_score(y_test, preds)), 4),
    "precision": round(float(precision_score(y_test, preds)), 4),
    "recall": round(float(recall_score(y_test, preds)), 4),
    "f1Score": round(float(f1_score(y_test, preds)), 4),
}
gpt2_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(gpt2_metrics)
results["gpt2_transformer"] = {
    "metrics": gpt2_metrics,
    "confusion": gpt2_confusion,
    "extra": {"algorithm": "GPT2ForSequenceClassification (full fine-tune)"},
}

## 15. ParsBERT — bidirectional encoder fine-tune

ParsBERT (`HooshvareLab/bert-fa-base-uncased`) brings the same pretrained
language knowledge as GPT-2 but with an architecture purpose-built for
classification: every token attends to the whole sequence and a dedicated
`[CLS]` token summarizes it. Reuses the normalized texts and loaders built
for GPT-2; the architecture is imported from `aitextd.architectures`.

In [ ]:
from transformers import AutoModel

PARSBERT_MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"

parsbert_tokenizer = AutoTokenizer.from_pretrained(PARSBERT_MODEL_NAME)
parsbert_backbone = AutoModel.from_pretrained(PARSBERT_MODEL_NAME)
parsbert_hidden_size = parsbert_backbone.config.hidden_size
print(f"hidden_size={parsbert_hidden_size}  vocab_size={parsbert_backbone.config.vocab_size}")

In [ ]:
PARSBERT_SEQ_LEN = 256
PARSBERT_BATCH_SIZE = 32
PARSBERT_MAX_EPOCHS = 10
PARSBERT_PATIENCE = 2
PARSBERT_LR_BACKBONE = 2e-5
PARSBERT_LR_HEAD = 1e-3
PARSBERT_WEIGHT_DECAY = 0.01

# Every encoder layer + the pooler is fine-tuned; the (much larger)
# word-embedding table stays frozen.
for p in parsbert_backbone.parameters():
    p.requires_grad = False
for layer in parsbert_backbone.encoder.layer:
    for p in layer.parameters():
        p.requires_grad = True
for p in parsbert_backbone.pooler.parameters():
    p.requires_grad = True

parsbert_model = ParsBERTClassifier(parsbert_backbone, parsbert_hidden_size).to(DEVICE)
parsbert_trainable = [p for p in parsbert_backbone.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in parsbert_trainable) + sum(
    p.numel() for p in parsbert_model.head.parameters()
)
print(f"Trainable parameters: {n_trainable:,}")

In [ ]:
def parsbert_encode(text_batch):
    enc = parsbert_tokenizer(
        list(text_batch),
        return_tensors="pt",
        truncation=True,
        max_length=PARSBERT_SEQ_LEN,
        padding="max_length",
    )
    return enc["input_ids"], enc["attention_mask"]


def parsbert_collate(batch):
    texts, labels = zip(*batch)
    input_ids, attention_mask = parsbert_encode(texts)
    return input_ids, attention_mask, torch.tensor(labels, dtype=torch.float32)


# Reuses the same character-normalized (not lemmatized) texts built for GPT-2.
parsbert_train_loader = DataLoader(
    TextLabelDataset(gpt2_train_texts, y_train), batch_size=PARSBERT_BATCH_SIZE,
    shuffle=True, collate_fn=parsbert_collate,
)
parsbert_val_loader = DataLoader(
    TextLabelDataset(gpt2_val_texts, y_val), batch_size=64, shuffle=False, collate_fn=parsbert_collate,
)
parsbert_test_loader = DataLoader(
    TextLabelDataset(gpt2_test_texts, y_test), batch_size=64, shuffle=False, collate_fn=parsbert_collate,
)

In [ ]:
parsbert_optimizer = torch.optim.AdamW(
    [
        {"params": parsbert_trainable, "lr": PARSBERT_LR_BACKBONE},
        {"params": parsbert_model.head.parameters(), "lr": PARSBERT_LR_HEAD},
    ],
    weight_decay=PARSBERT_WEIGHT_DECAY,
)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)


def parsbert_eval(loader):
    parsbert_model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, yb in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                p = torch.sigmoid(parsbert_model(input_ids.to(DEVICE), attention_mask.to(DEVICE)))
            probs.append(p.float().cpu().numpy())
            labels.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(labels)


def parsbert_score(labels, probs):
    preds = (probs >= 0.5).astype(int)
    return f1_score(labels, preds), accuracy_score(labels, preds)


best_val_f1, best_state, stale = -1.0, None, 0

for epoch in range(1, PARSBERT_MAX_EPOCHS + 1):
    t0 = time.time()
    parsbert_model.train()
    total_loss = 0.0
    for i, (input_ids, attention_mask, yb) in enumerate(parsbert_train_loader, start=1):
        input_ids, attention_mask, yb = (
            input_ids.to(DEVICE), attention_mask.to(DEVICE), yb.to(DEVICE)
        )
        parsbert_optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            loss = criterion(parsbert_model(input_ids, attention_mask), yb)
        scaler.scale(loss).backward()
        scaler.unscale_(parsbert_optimizer)
        torch.nn.utils.clip_grad_norm_(parsbert_model.parameters(), max_norm=1.0)
        scaler.step(parsbert_optimizer)
        scaler.update()
        total_loss += loss.item() * len(yb)
        if i % 50 == 0:
            print(f"  epoch {epoch} batch {i}/{len(parsbert_train_loader)} "
                  f"loss={loss.item():.4f} elapsed={time.time() - t0:.0f}s", flush=True)

    val_probs, val_labels = parsbert_eval(parsbert_val_loader)
    val_f1, val_acc = parsbert_score(val_labels, val_probs)
    print(f"Epoch {epoch}/{PARSBERT_MAX_EPOCHS}  loss={total_loss / len(gpt2_train_texts):.4f}  "
          f"val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({time.time() - t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1, stale = val_f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in parsbert_model.state_dict().items()}
    else:
        stale += 1
        if stale >= PARSBERT_PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {PARSBERT_PATIENCE} epochs).")
            break

parsbert_model.load_state_dict(best_state)
print(f"Best validation F1: {best_val_f1:.4f}")

In [ ]:
# Single held-out evaluation — the test slice is touched here and nowhere else.
probs, labels = parsbert_eval(parsbert_test_loader)
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(labels, preds)
tn, fp, fn, tp = cm.ravel()
parsbert_metrics = {
    "accuracy": round(float(accuracy_score(labels, preds)), 4),
    "precision": round(float(precision_score(labels, preds, zero_division=0)), 4),
    "recall": round(float(recall_score(labels, preds, zero_division=0)), 4),
    "f1Score": round(float(f1_score(labels, preds, zero_division=0)), 4),
}
parsbert_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(parsbert_metrics)
results["parsbert"] = {
    "metrics": parsbert_metrics,
    "confusion": parsbert_confusion,
    "extra": {"algorithm": "BertForSequenceClassification (full fine-tune)"},
}

In [ ]:
# Serving-contract check: rebuild the model exactly the way
# aitextd.loaders.load_parsbert_model does (BertConfig.from_dict ->
# AutoModel.from_config, i.e. with no pretrained weights) and assert the
# state_dict keys still line up. Catches architecture drift here rather than
# on the first live request after deployment.
from transformers import BertConfig

servable = ParsBERTClassifier(
    AutoModel.from_config(BertConfig.from_dict(parsbert_backbone.config.to_dict())),
    parsbert_hidden_size,
)
drift = set(parsbert_model.state_dict()) ^ set(servable.state_dict())
assert not drift, f"state_dict does not match the serving class: {drift}"
print("Serving contract OK —", len(parsbert_model.state_dict()), "tensors")

## 16. Summary (so far)

In [ ]:
for model_id, r in results.items():
    print(model_id, r["metrics"])

## 17. Save the models and build the `models.json` fragment

Writes every artifact trained above plus a JSON fragment using the exact ids
and `runtime` values `aitextd` dispatches on (`sklearn-tfidf`,
`pytorch-lstm`, `pytorch-gpt2`, `pytorch-parsbert`) — merging it needs no
backend change. To deploy: drop the files into the project's `models/`
directory and replace the matching entries in `models.json`.

Note: the TF-IDF vectorizer's vocabulary changed, so **every** `sklearn-tfidf`
model must come from this same run — don't mix an old model file with the new
`tfidf_vectorizer.joblib` or vice versa.

In [ ]:
joblib.dump(tfidf_vectorizer, f"{OUT_DIR}/tfidf_vectorizer.joblib")
joblib.dump(results["svm"]["model"], f"{OUT_DIR}/svModel.joblib")
joblib.dump(results["logreg"]["model"], f"{OUT_DIR}/logregModel.joblib")
joblib.dump(results["naive_bayes"]["model"], f"{OUT_DIR}/nbModel.joblib")
joblib.dump(results["random_forest"]["model"], f"{OUT_DIR}/rfModel.joblib")
joblib.dump(results["gradient_boosting"]["model"], f"{OUT_DIR}/gbModel.joblib")

torch.save(
    {
        "state_dict": lstm_model.state_dict(),
        "vocab": vocab,
        "config": {
            "embed_dim": EMBED_DIM,
            "hidden_dim": HIDDEN_DIM,
            "max_len": MAX_LEN,
            "pad_idx": PAD_IDX,
            "unk_idx": UNK_IDX,
        },
    },
    f"{OUT_DIR}/lstmModel.pt",
)

torch.save(
    {
        "state_dict": gpt2_model.state_dict(),
        "config": {
            "seq_len": GPT2_SEQ_LEN,
            "hidden_size": gpt2_hidden_size,
            "n_layer": gpt2_n_layer,
            "vocab_size": gpt2_backbone.config.vocab_size,
            "n_head": gpt2_backbone.config.n_head,
        },
    },
    f"{OUT_DIR}/gpt2ClassifierModel.pt",
)

# Export the fast tokenizers so the server can load them fully offline --
# aitextd.loaders prefers these folders automatically when present.
gpt2_tokenizer.save_pretrained(f"{OUT_DIR}/gpt2_tokenizer")

torch.save(
    {
        "state_dict": parsbert_model.state_dict(),
        "config": {
            "seq_len": PARSBERT_SEQ_LEN,
            "hidden_size": parsbert_hidden_size,
            # Full backbone config (num layers/heads/vocab size/etc.), so the
            # server can rebuild an identically-shaped BertModel purely from
            # this checkpoint -- no network access to the HF Hub needed at
            # inference time.
            "backbone_config": parsbert_backbone.config.to_dict(),
        },
        "metrics": parsbert_metrics,
    },
    f"{OUT_DIR}/parsbertClassifierModel.pt",
)
parsbert_tokenizer.save_pretrained(f"{OUT_DIR}/parsbert_tokenizer")

print("Saved model artifacts to", OUT_DIR)

In [ ]:
def training_info(feature_count, train_n=None):
    return {
        "totalSamples": len(train_idx) + len(val_idx) + len(test_idx),
        "trainSamples": len(train_idx) + len(val_idx) if train_n is None else train_n,
        "testSamples": len(test_idx),
        "featureCount": feature_count,
        "testSplitRatio": TEST_SIZE,
        "randomSeed": RANDOM_SEED,
        "dataset": "persian_ai_generated_final.csv + blogs.csv",
    }

vocab_size = len(tfidf_vectorizer.vocabulary_)

fragment = [
    {
        "id": "svm",
        "name": "SVM",
        "nameFa": "ماشین بردار پشتیبان",
        "description": "Linear Support Vector Machine (LinearSVC), hyperparameter-tuned via cross-validation on a unigram-to-trigram TF-IDF space, trained on Persian text to distinguish AI-generated content from human writing. Augmented with stylometric text-style features (see aitextd.features.STYLE_FEATURE_NAMES).",
        "descriptionFa": "ماشین بردار پشتیبان خطی با تنظیم هایپرپارامتر از طریق اعتبارسنجی متقابل روی فضای TF-IDF یک‌گرم تا سه‌گرم، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ویژگی‌های سبک‌شناسی متن.",
        "algorithm": "LinearSVC",
        "type": "ml",
        "version": "3.0",
        "filename": "svModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "styleFeatureCount": len(STYLE_FEATURE_NAMES),
        "metrics": results["svm"]["metrics"],
        "confusionMatrix": results["svm"]["confusion"],
        "training": training_info(vocab_size + len(STYLE_FEATURE_NAMES)),
    },
    {
        "id": "logreg",
        "name": "Logistic Regression",
        "nameFa": "رگرسیون لجستیک",
        "description": "Logistic Regression, hyperparameter-tuned via cross-validation on a unigram-to-trigram TF-IDF space, trained on Persian text to distinguish AI-generated content from human writing. Augmented with stylometric text-style features (see aitextd.features.STYLE_FEATURE_NAMES).",
        "descriptionFa": "رگرسیون لجستیک با تنظیم هایپرپارامتر از طریق اعتبارسنجی متقابل روی فضای TF-IDF یک‌گرم تا سه‌گرم، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ویژگی‌های سبک‌شناسی متن.",
        "algorithm": "LogisticRegression",
        "type": "ml",
        "version": "3.0",
        "filename": "logregModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "styleFeatureCount": len(STYLE_FEATURE_NAMES),
        "metrics": results["logreg"]["metrics"],
        "confusionMatrix": results["logreg"]["confusion"],
        "training": training_info(vocab_size + len(STYLE_FEATURE_NAMES)),
    },
    {
        "id": "naive_bayes",
        "name": "Naive Bayes",
        "nameFa": "بیز ساده",
        "description": "Multinomial Naive Bayes, alpha-tuned via cross-validation on a unigram-to-trigram TF-IDF space, trained on Persian text to distinguish AI-generated content from human writing. Augmented with stylometric text-style features (see aitextd.features.STYLE_FEATURE_NAMES).",
        "descriptionFa": "بیز ساده چندجمله‌ای با تنظیم آلفا از طریق اعتبارسنجی متقابل روی فضای TF-IDF یک‌گرم تا سه‌گرم، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ویژگی‌های سبک‌شناسی متن.",
        "algorithm": "MultinomialNB",
        "type": "ml",
        "version": "3.0",
        "filename": "nbModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "styleFeatureCount": len(STYLE_FEATURE_NAMES),
        "metrics": results["naive_bayes"]["metrics"],
        "confusionMatrix": results["naive_bayes"]["confusion"],
        "training": training_info(vocab_size + len(STYLE_FEATURE_NAMES)),
    },
    {
        "id": "random_forest",
        "name": "Random Forest",
        "nameFa": "جنگل تصادفی",
        "description": "Random Forest, hyperparameter-tuned (including bounded tree depth to fight overfitting) via cross-validation on the top 30,000 chi2-selected TF-IDF features, trained on Persian text to distinguish AI-generated content from human writing. Augmented with stylometric text-style features (see aitextd.features.STYLE_FEATURE_NAMES).",
        "descriptionFa": "جنگل تصادفی با تنظیم هایپرپارامتر (از جمله محدودسازی عمق درخت برای کاهش بیش‌برازش) روی ۳۰٬۰۰۰ ویژگی برتر TF-IDF (انتخاب‌شده با chi2)، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ویژگی‌های سبک‌شناسی متن.",
        "algorithm": "RandomForestClassifier",
        "type": "ml",
        "version": "3.0",
        "filename": "rfModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "styleFeatureCount": len(STYLE_FEATURE_NAMES),
        "metrics": results["random_forest"]["metrics"],
        "confusionMatrix": results["random_forest"]["confusion"],
        "training": training_info(results["random_forest"]["extra"]["featureCount"]),
    },
    {
        "id": "gradient_boosting",
        "name": "Gradient Boosting",
        "nameFa": "گرادیان بوستینگ",
        "description": "Gradient Boosting, hyperparameter-tuned via cross-validation on the top 30,000 chi2-selected TF-IDF features, trained on Persian text to distinguish AI-generated content from human writing. Augmented with stylometric text-style features (see aitextd.features.STYLE_FEATURE_NAMES).",
        "descriptionFa": "گرادیان بوستینگ با تنظیم هایپرپارامتر از طریق اعتبارسنجی روی ۳۰٬۰۰۰ ویژگی برتر TF-IDF (انتخاب‌شده با chi2)، آموزش‌دیده برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان. همراه با ویژگی‌های سبک‌شناسی متن.",
        "algorithm": "GradientBoostingClassifier",
        "type": "ml",
        "version": "3.0",
        "filename": "gbModel.joblib",
        "runtime": "sklearn-tfidf",
        "ensembleWeight": 1.0,
        "vectorizer": "tfidf_vectorizer.joblib",
        "styleFeatureCount": len(STYLE_FEATURE_NAMES),
        "metrics": results["gradient_boosting"]["metrics"],
        "confusionMatrix": results["gradient_boosting"]["confusion"],
        "training": training_info(results["gradient_boosting"]["extra"]["featureCount"]),
    },
    {
        "id": "bilstm",
        "name": "BiLSTM",
        "nameFa": "شبکه دوسویه LSTM",
        "description": "A bidirectional LSTM with its own learned word embeddings, trained from scratch on GPU with early stopping against a held-out validation set (not the test set), on lemmatized Persian text, to distinguish AI-generated content from human writing.",
        "descriptionFa": "شبکه عصبی LSTM دوسویه با embedding آموخته‌شده اختصاصی، آموزش‌دیده از ابتدا روی GPU با early stopping بر پایه مجموعه اعتبارسنجی جداگانه (نه مجموعه آزمون)، روی متن فارسی نرمال‌شده و ریشه‌یابی‌شده، برای تشخیص متن تولیدشده توسط هوش مصنوعی از نوشته انسان.",
        "algorithm": "BiLSTM",
        "type": "dl",
        "version": "2.0",
        "filename": "lstmModel.pt",
        "runtime": "pytorch-lstm",
        "ensembleWeight": 1.5,
        "metrics": results["bilstm"]["metrics"],
        "confusionMatrix": results["bilstm"]["confusion"],
        "training": training_info(results["bilstm"]["extra"]["vocab_size"]),
    },
    {
        "id": "gpt2_transformer",
        "name": "GPT-2 Transformer (fine-tuned)",
        "nameFa": "ترنسفورمر GPT-2 (فاین‌تیون‌شده)",
        "description": "HooshvareLab/gpt2-fa (a 12-layer, 124M-parameter Persian GPT-2) fully fine-tuned (all 12 transformer blocks) on the full training set with 128 tokens of context, GPU + mixed precision, and validation-based early stopping. Text is normalized (not lemmatized) before tokenizing with GPT-2's own fast tokenizer, to match its pretraining distribution.",
        "descriptionFa": "مدل GPT-2 فارسی (HooshvareLab/gpt2-fa) با ۱۲ لایه و ۱۲۴ میلیون پارامتر که تمام ۱۲ بلوک ترنسفورمر آن روی کل مجموعه آموزشی با ۱۲۸ توکن زمینه، GPU و دقت مختلط، و early stopping بر پایه مجموعه اعتبارسنجی فاین‌تیون شده است. متن پیش از توکن‌سازی با توکنایزر سریع خود GPT-2 تنها نرمال‌سازی می‌شود (نه ریشه‌یابی) تا با توزیع پیش‌آموزش مدل سازگار بماند.",
        "algorithm": "GPT2ForSequenceClassification (full fine-tune)",
        "type": "dl",
        "version": "2.0",
        "filename": "gpt2ClassifierModel.pt",
        "runtime": "pytorch-gpt2",
        "ensembleWeight": 3.0,
        "metrics": results["gpt2_transformer"]["metrics"],
        "confusionMatrix": results["gpt2_transformer"]["confusion"],
        "training": training_info(gpt2_hidden_size),
    },
    {
        "id": "parsbert",
        "name": "ParsBERT Transformer (fine-tuned)",
        "nameFa": "ترنسفورمر ParsBERT (فاین‌تیون‌شده)",
        "description": "HooshvareLab/bert-fa-base-uncased (a 12-layer, ~110M-parameter Persian BERT) fully fine-tuned (all 12 encoder layers) on the full training set with 256 tokens of context, GPU + mixed precision, and validation-based early stopping. Uses the bidirectional [CLS] token representation instead of GPT-2's last-token pooling, and its own WordPiece tokenizer on lightly normalized (not lemmatized) text.",
        "descriptionFa": "مدل ParsBERT فارسی (HooshvareLab/bert-fa-base-uncased) با ۱۲ لایه و حدود ۱۱۰ میلیون پارامتر که تمام ۱۲ لایهٔ رمزگذار آن روی کل مجموعه آموزشی با ۲۵۶ توکن زمینه، GPU و دقت مختلط، و early stopping بر پایه مجموعه اعتبارسنجی فاین‌تیون شده است. برخلاف GPT-2، از بازنمایی دوسویهٔ توکن [CLS] و توکن‌ساز WordPiece اختصاصی خود روی متن نرمال‌شده (نه ریشه‌یابی‌شده) استفاده می‌کند.",
        "algorithm": "BertModel + linear head on [CLS]",
        "type": "dl",
        "version": "2.0",
        "filename": "parsbertClassifierModel.pt",
        "runtime": "pytorch-parsbert",
        "ensembleWeight": 2.0,
        "metrics": results["parsbert"]["metrics"],
        "confusionMatrix": results["parsbert"]["confusion"],
        "training": training_info(parsbert_hidden_size),
    },
]

with open(f"{OUT_DIR}/models_colab_update.json", "w", encoding="utf-8") as f:
    json.dump(fragment, f, ensure_ascii=False, indent=2)

print("Wrote models_colab_update.json with ids:", [e["id"] for e in fragment])

## 18. System One — the typed decision data

The dataset only carries a binary label, but System One answers *three* typed
questions, so soft targets are distilled from the deployed TF-IDF +
Logistic Regression pipeline. The target-shaping functions and distillation
weights are imported from `training/prepare_system_one_data.py`, and the question
schema from `aitextd.constants` — the same code the local pipeline uses.

Per row:

- **choice** — ground-truth one-hot blended 0.8/0.2 with the teacher's
  probability, clipped to $[0.02, 0.98]$ so no target is ever exactly certain.
- **score** — a discretized Gaussian over the 5 ordinal levels centred at
  $4 \times$ the blended AI probability (teaches "between levels" answers).
- **noul** — 1 where the teacher itself is uncertain (probability in
  $[0.15, 0.85]$): those are the only rows where a human reviewer genuinely
  adds information.

In [ ]:
teacher = joblib.load(os.path.join(DRIVE_DIR, "logregModel.joblib"))

STYLE_CACHE = os.path.join(DRIVE_DIR, "system_one_style_features.joblib")
# Only the rows System One actually trains/evaluates on need preprocessing.
needed = np.concatenate([train_idx, test_idx])
cleaned = {int(i): " ".join(tokenized[i]) for i in needed}

try:
    payload = joblib.load(STYLE_CACHE)
    assert payload["version"] == STYLE_FEATURES_VERSION
    style_matrix = payload["matrix"]
    assert style_matrix.shape == (len(needed), len(STYLE_FEATURE_NAMES))
    print(f"Loaded cached style features for {style_matrix.shape[0]} rows.")
except (FileNotFoundError, KeyError, TypeError, IndexError, AssertionError):
    print(f"Computing style features for {len(needed)} rows (single-process, "
          "the vectorizer is shared)...")
    style_matrix = np.array([
        [compute_style_features(str(df["text"].iloc[i]), cleaned[int(i)], tfidf_vectorizer)[name]
         for name in STYLE_FEATURE_NAMES]
        for i in tqdm(needed, desc="style features")
    ])
    joblib.dump({"version": STYLE_FEATURES_VERSION, "matrix": style_matrix}, STYLE_CACHE)
    print("Cached to Drive for next time.")

tfidf = tfidf_vectorizer.transform([cleaned[int(i)] for i in needed])
teacher_style_cols = style_matrix[:, : teacher.n_features_in_ - tfidf.shape[1]]
teacher_features = hstack([tfidf, csr_matrix(teacher_style_cols)]).tocsr()
teacher_prob = teacher.predict_proba(teacher_features)[:, 1]
print(f"Teacher scored {len(needed)} rows; mean P(AI) = {teacher_prob.mean():.4f}")

In [ ]:
def build_system_one_records(indices, split_name, position):
    records = []
    for k, i in enumerate(indices):
        label = int(y_all[i])
        source_t, t_ai = choice_target(label, teacher_prob[position[k]])
        records.append({
            "id": f"{split_name}-{i:06d}",
            "split": split_name,
            "label": label,
            "text": str(df["text"].iloc[i]),
            "targets": {
                "source": source_t,
                "ai_level": score_target(t_ai),
                "needs_review": noul_target(teacher_prob[position[k]]),
            },
            "teacherAiProbability": round(float(teacher_prob[position[k]]), 4),
        })
    return records


train_val_records = build_system_one_records(train_idx, "train", range(len(train_idx)))
test_records = build_system_one_records(
    test_idx, "test", range(len(train_idx), len(train_idx) + len(test_idx))
)

# Carve the validation slice out of the training records, same stratified rule.
train_positions, val_positions = train_test_split(
    np.arange(len(train_val_records)),
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=[r["label"] for r in train_val_records],
)
train_records = [train_val_records[p] for p in train_positions]
val_records = [train_val_records[p] for p in val_positions]

for name, recs in [("train", train_records), ("val", val_records), ("test", test_records)]:
    noul_pos = sum(r["targets"]["needs_review"] for r in recs)
    print(f"{name}: {len(recs)} records | needs_review targets={int(noul_pos)}")

print(json.dumps(
    {**train_records[0], "text": train_records[0]["text"][:100] + "..."},
    ensure_ascii=False, indent=2,
))

## 19. System One — the typed decision model

> **⚠️ Legacy recipe.** This section reproduces the legacy GPT-2-backed
> System One checkpoint format (pooled on the last real token, no
> `backbone_config`). The currently-deployed `systemOneModel.pt` is the
> ParsBERT-backed v2 distilled from the GPT-2 classifier — train it locally
> with `python training/train_system_one.py` (see the report). Only follow
> this section if you specifically want the legacy format.

A pretrained-transformer backbone pooled into three small linear heads,
answered in one forward pass. The architecture is imported from
`aitextd.architectures` (`SystemOneDecisionModel`), which the serving loader also
uses — the serving-contract check below proves the state_dicts line up.

In [ ]:
from transformers import GPT2LMHeadModel

gpt2_lm = GPT2LMHeadModel.from_pretrained(GPT2_MODEL_NAME)
system_one_backbone = gpt2_lm.transformer
system_one_hidden_size = system_one_backbone.config.n_embd
system_one_n_layer = system_one_backbone.config.n_layer
print(f"n_layer={system_one_n_layer}  n_embd={system_one_hidden_size}")

SYSTEM_ONE_SEQ_LEN = 128
SYSTEM_ONE_BATCH_SIZE = 32
SYSTEM_ONE_MAX_EPOCHS = 10
SYSTEM_ONE_PATIENCE = 2
SYSTEM_ONE_LR_BACKBONE = 2e-5
SYSTEM_ONE_LR_HEAD = 1e-3
SYSTEM_ONE_UNFROZEN_BLOCKS = 4

CHOICE_WEIGHT, SCORE_WEIGHT, NOUL_WEIGHT = 1.0, 0.5, 0.5

# Only the last 4 of 12 transformer blocks plus the final LayerNorm are
# unfrozen: 21k rows is not enough to move a pretrained model much.
for p in system_one_backbone.parameters():
    p.requires_grad = False
for block in system_one_backbone.h[system_one_n_layer - SYSTEM_ONE_UNFROZEN_BLOCKS:]:
    for p in block.parameters():
        p.requires_grad = True
for p in system_one_backbone.ln_f.parameters():
    p.requires_grad = True

system_one_model = SystemOneDecisionModel(
    system_one_backbone, system_one_hidden_size,
    n_choice=len(SYSTEM_ONE_CHOICE_CANDIDATES), n_score_levels=len(SYSTEM_ONE_SCORE_LEVELS),
    pool_on_cls=False,
).to(DEVICE)

system_one_optimizer = torch.optim.AdamW([
    {"params": [p for p in system_one_backbone.parameters() if p.requires_grad], "lr": SYSTEM_ONE_LR_BACKBONE},
    {"params": system_one_model.choice_head.parameters(), "lr": SYSTEM_ONE_LR_HEAD},
    {"params": system_one_model.score_head.parameters(), "lr": SYSTEM_ONE_LR_HEAD},
    {"params": system_one_model.noul_head.parameters(), "lr": SYSTEM_ONE_LR_HEAD},
])
print("System One model ready.")

In [ ]:
def system_one_encode(text_batch):
    enc = gpt2_tokenizer(
        list(text_batch),
        return_tensors="pt",
        truncation=True,
        max_length=SYSTEM_ONE_SEQ_LEN,
        padding="max_length",
    )
    return enc["input_ids"], enc["attention_mask"]


def system_one_collate(batch):
    texts, targets, labels = zip(*batch)
    input_ids, attention_mask = system_one_encode(texts)
    return (
        input_ids,
        attention_mask,
        torch.tensor(
            [[t["source"][c] for c in SYSTEM_ONE_CHOICE_CANDIDATES] for t in targets],
            dtype=torch.float32,
        ),
        torch.tensor([t["ai_level"] for t in targets], dtype=torch.float32),
        torch.tensor([t["needs_review"] for t in targets], dtype=torch.float32),
        torch.tensor(labels, dtype=torch.float32),
    )


def to_system_one_dataset(records):
    return [(r["text"], r["targets"], r["label"]) for r in records]


system_one_train_loader = DataLoader(
    to_system_one_dataset(train_records), batch_size=SYSTEM_ONE_BATCH_SIZE, shuffle=True, collate_fn=system_one_collate,
)
system_one_val_loader = DataLoader(
    to_system_one_dataset(val_records), batch_size=64, shuffle=False, collate_fn=system_one_collate,
)
system_one_test_loader = DataLoader(
    to_system_one_dataset(test_records), batch_size=64, shuffle=False, collate_fn=system_one_collate,
)


def soft_cross_entropy(logits, target_probs):
    return -(target_probs * torch.log_softmax(logits, dim=-1)).sum(dim=-1).mean()


def system_one_loss(choice_logits, score_logits, noul_logit, choice_t, score_t, noul_t):
    return (
        CHOICE_WEIGHT * soft_cross_entropy(choice_logits, choice_t)
        + SCORE_WEIGHT * soft_cross_entropy(score_logits, score_t)
        + NOUL_WEIGHT * nn.functional.binary_cross_entropy_with_logits(noul_logit, noul_t)
    )


@torch.no_grad()
def system_one_eval(loader):
    system_one_model.eval()
    choice_p, score_p, noul_p, labels = [], [], [], []
    for input_ids, attention_mask, _, _, _, yb in loader:
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            cl, sl, nl = system_one_model(input_ids.to(DEVICE), attention_mask.to(DEVICE))
        choice_p.append(torch.softmax(cl, dim=-1).float().cpu().numpy())
        score_p.append(torch.softmax(sl, dim=-1).float().cpu().numpy())
        noul_p.append(torch.sigmoid(nl).float().cpu().numpy())
        labels.append(yb.numpy())
    return (np.concatenate(choice_p), np.concatenate(score_p),
            np.concatenate(noul_p), np.concatenate(labels))

In [ ]:
best_val_f1, best_state, stale = -1.0, None, 0

for epoch in range(1, SYSTEM_ONE_MAX_EPOCHS + 1):
    t0 = time.time()
    system_one_model.train()
    total_loss = 0.0
    for i, batch in enumerate(system_one_train_loader, start=1):
        input_ids, attention_mask, choice_t, score_t, noul_t, _ = [
            t.to(DEVICE) for t in batch
        ]
        system_one_optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            cl, sl, nl = system_one_model(input_ids, attention_mask)
            loss = system_one_loss(cl, sl, nl, choice_t, score_t, noul_t)
        scaler.scale(loss).backward()
        scaler.unscale_(system_one_optimizer)
        torch.nn.utils.clip_grad_norm_(system_one_model.parameters(), max_norm=1.0)
        scaler.step(system_one_optimizer)
        scaler.update()
        total_loss += loss.item() * len(input_ids)
        if i % 50 == 0:
            print(f"  epoch {epoch} batch {i}/{len(system_one_train_loader)} "
                  f"loss={loss.item():.4f} elapsed={time.time() - t0:.0f}s", flush=True)

    val_choice_p, _, _, val_labels = system_one_eval(system_one_val_loader)
    val_f1 = f1_score(val_labels, val_choice_p.argmax(axis=1))
    print(f"Epoch {epoch}/{SYSTEM_ONE_MAX_EPOCHS}  loss={total_loss / len(train_records):.4f}  "
          f"val_f1={val_f1:.4f}  ({time.time() - t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1, stale = val_f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in system_one_model.state_dict().items()}
    else:
        stale += 1
        if stale >= SYSTEM_ONE_PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {SYSTEM_ONE_PATIENCE} epochs).")
            break

system_one_model.load_state_dict(best_state)
print(f"Best validation F1: {best_val_f1:.4f}")

In [ ]:
def expected_calibration_error(probs, labels, n_bins=15):
    confidence = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)


choice_p, score_p, noul_p, test_labels = system_one_eval(system_one_test_loader)
preds = choice_p.argmax(axis=1)
cm = confusion_matrix(test_labels, preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

score_t_test = np.array([r["targets"]["ai_level"] for r in test_records])
noul_t_test = np.array([r["targets"]["needs_review"] for r in test_records])

system_one_metrics = {
    "accuracy": round(float(accuracy_score(test_labels, preds)), 4),
    "precision": round(float(precision_score(test_labels, preds, zero_division=0)), 4),
    "recall": round(float(recall_score(test_labels, preds, zero_division=0)), 4),
    "f1Score": round(float(f1_score(test_labels, preds, zero_division=0)), 4),
    "scoreMae": round(float(mean_absolute_error(
        score_t_test @ np.arange(len(SYSTEM_ONE_SCORE_LEVELS)),
        score_p @ np.arange(len(SYSTEM_ONE_SCORE_LEVELS)),
    )), 4),
    "choiceEce": round(expected_calibration_error(choice_p, test_labels), 4),
}
# Both noul classes must be present for AUC; on a near-separable dataset the
# uncertain band can be nearly empty.
if len(np.unique(noul_t_test)) > 1:
    system_one_metrics["noulAuc"] = round(float(roc_auc_score(noul_t_test, noul_p)), 4)
    system_one_metrics["noulAveragePrecision"] = round(
        float(average_precision_score(noul_t_test, noul_p)), 4
    )

system_one_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(system_one_metrics)

In [ ]:
# Serving-contract check: the serving loader (aitextd.loaders.load_system_one_model)
# builds the backbone from the local gpt2-fa.mdl pickle and then loads this
# state_dict on top, so the two must expose the same tensor names.
print(f"state_dict tensors: {len(system_one_model.state_dict())}")
print(f"backbone tensors:   {len(system_one_backbone.state_dict())} + 3 heads (choice/score/noul)")
assert any(k.startswith("backbone.h.11.") for k in system_one_model.state_dict())
assert "choice_head.weight" in system_one_model.state_dict()
assert "score_head.weight" in system_one_model.state_dict()
assert "noul_head.weight" in system_one_model.state_dict()
print("Serving contract OK.")

In [ ]:
PROCEED_THRESHOLD = 0.8
NEEDS_REVIEW_THRESHOLD = 0.6

torch.save(
    {
        "state_dict": system_one_model.state_dict(),
        "config": {
            "seq_len": SYSTEM_ONE_SEQ_LEN,
            "hidden_size": system_one_hidden_size,
            "n_choice": len(SYSTEM_ONE_CHOICE_CANDIDATES),
            "n_score_levels": len(SYSTEM_ONE_SCORE_LEVELS),
            "choice_candidates": SYSTEM_ONE_CHOICE_CANDIDATES,
            "score_levels": SYSTEM_ONE_SCORE_LEVELS,
            "thresholds": {
                "proceed": PROCEED_THRESHOLD,
                "needsReview": NEEDS_REVIEW_THRESHOLD,
            },
        },
        "metrics": system_one_metrics,
    },
    f"{OUT_DIR}/systemOneModel.pt",
)
print("System One artifacts written to", OUT_DIR)

## 20. Add the System One entry to the `models.json` fragment

`ensembleWeight` is carried over from the current `models.json` when that
file is on Drive — the ensemble in `aitextd.store` skips any model weighted 0,
and silently changing that here would change every prediction the API
returns. The appended `system_one` entry describes the legacy-format
checkpoint trained in section 19 — skip this cell if you did not retrain System One.

In [ ]:
MODELS_JSON = os.path.join(DRIVE_DIR, "models.json")
existing = {}
if os.path.isfile(MODELS_JSON):
    with open(MODELS_JSON, encoding="utf-8") as f:
        existing = {m["id"]: m for m in json.load(f)}
else:
    print(f"No models.json in {DRIVE_DIR}; defaulting ensembleWeight to 1.0.")


def weight_for(model_id, default):
    return float(existing.get(model_id, {}).get("ensembleWeight", default))


n_total = len(train_records) + len(val_records) + len(test_records)

fragment.append(
    {
        "id": "system_one",
        "name": "System One System One (Decision Head)",
        "nameFa": "مدل تصمیم سیستم-یک (System One)",
        "description": (
            "System One-style 'System One' decision model: the Persian GPT-2 backbone "
            "(HooshvareLab/gpt2-fa, last 4 blocks fine-tuned) pooled into three typed "
            "decision heads answered in a single forward pass -- a choice question "
            "(human vs AI, as a probability distribution), a score question "
            "(machine-likeness on a 5-level scale, answered as a probability-weighted "
            "value between levels) and a noul question (probability that human review "
            "is needed). Trained on soft targets distilled from the deployed TF-IDF + "
            "Logistic Regression pipeline."
        ),
        "descriptionFa": (
            "مدل تصمیم «سیستم-یک» به سبک System One: بک‌بون GPT-2 فارسی "
            "(HooshvareLab/gpt2-fa با فاین‌تیون ۴ بلوک آخر) که بازنمایی تجمیع‌شده آن "
            "به سه سر تصمیم نوع‌بندی‌شده می‌رود و همه در یک گذر رو به جلو پاسخ داده "
            "می‌شوند — سؤال انتخاب (انسان در برابر هوش مصنوعی به شکل توزیع احتمال)، "
            "سؤال امتیاز (ماشین‌ماندبودن روی طیف ۵ سطحی با مقدار وزن‌دار احتمالاتی که "
            "می‌تواند بین سطوح بیفتد) و سؤانول (احتمال نیاز به بازبینی انسانی). "
            "آموزش با هدف‌های نرم تقطیرشده از خط لولهٔ TF-IDF + رگرسیون لجستیکِ مستقر."
        ),
        "algorithm": "GPT2 + typed decision heads (choice/score/noul)",
        "type": "dl",
        "version": "2.0",
        "filename": "systemOneModel.pt",
        "runtime": "pytorch-system-one",
        "ensembleWeight": weight_for("system_one", 1.0),
        "metrics": system_one_metrics,
        "confusionMatrix": system_one_confusion,
        "training": {
            "totalSamples": n_total,
            "trainSamples": len(train_records),
            "testSamples": len(test_records),
            "featureCount": system_one_hidden_size,
            "testSplitRatio": len(test_records) / n_total,
            "randomSeed": RANDOM_SEED,
            "dataset": "processedDataset.csv (System One records, teacher-distilled soft targets)",
        },
    }
)

with open(f"{OUT_DIR}/models_colab_update.json", "w", encoding="utf-8") as f:
    json.dump(fragment, f, ensure_ascii=False, indent=2)
print("Wrote models_colab_update.json with ids:", [e["id"] for e in fragment])

print("\n=== Final summary ===")
for name, metrics in results.items():
    print(f"{name}: {metrics['metrics']}")
print(f"system_one: {system_one_metrics}")

## 21. Zip and download

In [ ]:
zip_path = shutil.make_archive("/content/retrained_models", "zip", OUT_DIR)

total_mb = 0
for root, _, files in os.walk(OUT_DIR):
    for name in files:
        path = os.path.join(root, name)
        size_mb = os.path.getsize(path) / 1e6
        total_mb += size_mb
        print(f"{size_mb:9.1f} MB  {os.path.relpath(path, OUT_DIR)}")
print(f"{total_mb:9.1f} MB  total")

from google.colab import files

files.download(zip_path)